# Saving a Trained Model to Drive and Scoring It Later

The `train_*.ipynb` notebooks end by zipping the checkpoint and pushing it through a browser
download. That works, but a ModernBERT or t5-base save is close to a gigabyte, and a stalled
download loses the run. Google Drive is the better destination: it survives the runtime, it is
already attached to the account you are running Colab under, and reloading from it in a new session
is one `from_pretrained` call.

This notebook is in two halves.

**Part 1** runs at the end of a training session, in the same runtime that still holds `trainer` and
`tokenizer`. It writes the model and its tokenizer to Drive.

**Part 2** runs in a *fresh* runtime. It reloads them and reproduces the test metrics. Deleting the
runtime in between is the point — it is the only way to know the save was complete rather than the
session quietly filling in the gaps.

ModernBERT is the worked example. The last section shows T5 and GPT-2, which differ only in which
`run_*` module they import from.

## Part 1 — Save

Run these cells after `trainer.train()` in `train_bert.ipynb`, without restarting anything.

Mounting prompts for account access and prints an authorisation link. `MyDrive` is the root of the
Drive you see in the browser.

In [ ]:
from google.colab import drive


drive.mount("/content/drive")

### Save the model and the tokenizer

`trainer.save_model` writes the weights and `config.json` — inference files only, no optimizer
state. `load_best_model_at_end=True` is a default in `CLASSIFICATION`, so the model in memory at
this point is the best epoch rather than the last one, and that is what gets written.

The tokenizer line is redundant on recent `transformers`, which already persists
`trainer.processing_class` inside `save_model`. Writing it explicitly costs nothing and does not
depend on that behaviour holding.

The test split is not saved. Part 2 rebuilds it — see the note there on what that assumes.

In [ ]:
from pathlib import Path


SAVE_DIR = Path("/content/drive/MyDrive/bert-t5-gpt2/bert")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

### What landed

Five files:

- `config.json` — architecture and hyperparameters, and the `id2label` mapping `build_trainer`
  passed in. That mapping travelling with the model is why the reloaded classifier says `Positive`
  instead of `LABEL_2`.
- `model.safetensors` — the weights.
- `tokenizer.json`, `tokenizer_config.json`, `special_tokens_map.json` — the vocabulary and the
  special-token setup.

Drive writes are slow. A ModernBERT-base save is a few hundred megabytes and takes a minute or two
to finish flushing; wait for the cell to return before disconnecting.

In [ ]:
for path in sorted(SAVE_DIR.iterdir()):
    size = sum(item.stat().st_size for item in path.rglob("*")) if path.is_dir() else path.stat().st_size
    print(f"{path.name:<28} {size / 1e6:>9.2f} MB")

### Save `SAVE_DIR`, not `training_args.output_dir`

`CLASSIFICATION` sets `save_strategy="epoch"` and `save_total_limit=2`, so
`artifacts/bert/` also holds two `checkpoint-N/` directories. Those carry optimizer and scheduler
state — roughly three times the model size each — because they exist to resume an interrupted run,
not to serve predictions.

Copying that folder wholesale to Drive uploads several gigabytes you will never load. `save_model`
into a clean directory writes only what scoring needs.

## Part 2 — Score in a fresh runtime

Before running anything below: **Runtime -> Disconnect and delete runtime**, then reconnect.

This is not ceremony. If `model` and `test` are still in memory from Part 1, a save that silently
dropped a file would still appear to work, and you would find out weeks later. A deleted runtime
makes the reload prove itself.

### Reinstall the package

The install has to be editable for the same reason it does in the training notebooks:
`headlines/config.py` computes `PROJECT_ROOT = Path(__file__).resolve().parents[2]`, so the data
paths resolve relative to wherever `config.py` physically sits. A regular install copies it into
`site-packages`, where `parents[2]` points at nothing useful.

Scoring the saved split does not actually read the CSVs, but the fallback in the split cell below
does, and so does anything you go on to do with `build_datasets`.

In [ ]:
![ -d /content/Bert-T5-GPT2 ] || git clone https://github.com/nickkats1/Bert-T5-GPT2

In [ ]:
%pip install -q -e /content/Bert-T5-GPT2

### Re-mount Drive

New runtime, new mount. `SAVE_DIR` has to be defined again too — it was a Python variable, and that
runtime is gone.

In [ ]:
from pathlib import Path

from google.colab import drive


drive.mount("/content/drive")
SAVE_DIR = Path("/content/drive/MyDrive/bert-t5-gpt2/bert")

### Rebuild the trainer from the saved folder

`model_name_or_path` takes a local directory anywhere a Hub name would go, so pointing
`ClassificationModelArguments` at `SAVE_DIR` gives `build_trainer` the trained weights instead of
the Hub checkpoint. Everything else — the tokenizer, the label maps, the collator,
`compute_classification_metrics`, the precision policy in `resolve_precision` — is wired exactly as
it was during training, which is the point: a hand-built eval `Trainer` is where the two paths
silently drift apart.

`build_trainer` rebuilds the splits rather than reading a saved copy. `split_frame` is deterministic,
so a default `ClassificationDataArguments()` reproduces the identical `train`/`validation`/`test`
partition in any session — **as long as `seed`, `test_size`, `val_test_ratio`, and the CSV are all
untouched.** Change the seed, widen the test fraction, or add rows to `guardian_headlines.csv`, and
rows the model trained on drift into the partition you are calling held out. The score goes up and
nothing warns you. If you expect the data to move, save `trainer.test_dataset` with `save_to_disk`
in Part 1 and score that instead.

Nothing here reaches the network — the weights come off Drive.

In [ ]:
from headlines.bert.config import CLASSIFICATION, ClassificationDataArguments, ClassificationModelArguments
from headlines.bert.train import build_trainer
from headlines.config import training_arguments


trainer = build_trainer(
    ClassificationModelArguments(model_name_or_path=str(SAVE_DIR)),
    ClassificationDataArguments(),
    training_arguments(CLASSIFICATION, output_dir="/content/eval", per_device_eval_batch_size=32),
)
tokenizer = trainer.processing_class

print(trainer.model.config.id2label)
print(f"{sum(parameter.numel() for parameter in trainer.model.parameters()):,} parameters")
print(f"{len(trainer.test_dataset):,} held-out test rows")

metrics = trainer.evaluate(trainer.test_dataset, metric_key_prefix="test")
for name, value in metrics.items():
    if isinstance(value, float):
        print(f"{name:<25} {value:.4f}")

### Confirm the round trip

Compare `test_f1_weighted` and `test_f1_macro` above against what `train_bert.ipynb` printed. They
should match to four decimals: same weights, same split, same metric function, and
`resolve_precision` makes the same bf16-or-fp32 decision here that it made during training. A large
difference means the model being scored is not the model you trained.

The gap between the weighted and macro numbers is the same story as in training: weighted F1 is
flattered by the Neutral majority, macro F1 weights all three classes equally.

In [ ]:
import numpy as np
import seaborn as sns
from matplotlib import pyplot as plt
from sklearn.metrics import confusion_matrix

from headlines.bert.metrics import score_predictions
from headlines.bert.utils.labeling import ID2LABEL


prediction = trainer.predict(trainer.test_dataset)
y_pred = np.argmax(prediction.predictions, axis=-1)

for name, value in score_predictions(prediction.label_ids, y_pred).items():
    print(f"{name:<20} {value:.4f}")

class_names = [ID2LABEL[index] for index in sorted(ID2LABEL)]
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    confusion_matrix(prediction.label_ids, y_pred),
    annot=True,
    fmt="d",
    cmap="viridis",
    xticklabels=class_names,
    yticklabels=class_names,
    ax=ax,
)
ax.set_xlabel("predicted")
ax.set_ylabel("actual")
ax.set_title("Reloaded model, held-out test split")
plt.show()

## T5 and GPT-2

Part 1 is identical for all three — `save_model` and `save_pretrained` into a different `SAVE_DIR`.
Part 2 is identical in shape too. Each task swaps in its own three imports and its own config dict:

| | package | training arguments | config dict |
| --- | --- | --- | --- |
| BERT | `headlines.bert` | `training_arguments` | `CLASSIFICATION` |
| T5 | `headlines.t5` | `seq2seq_arguments` | `SUMMARIZATION` |
| GPT-2 | `headlines.gpt2` | `training_arguments` | `CLM` |

The collator, the metrics, and `predict_with_generate` all come along with the config dict, so
nothing about the scoring harness has to be restated per model. T5 in particular needs
`predict_with_generate=True` or `compute_metrics` receives logits instead of token ids and ROUGE
collapses — `SUMMARIZATION` already sets it.

In [ ]:
from headlines.config import seq2seq_arguments
from headlines.t5.config import SUMMARIZATION, SummarizationDataArguments, SummarizationModelArguments
from headlines.t5.train import build_trainer as build_t5_trainer


T5_DIR = Path("/content/drive/MyDrive/bert-t5-gpt2/t5")

t5_trainer = build_t5_trainer(
    SummarizationModelArguments(model_name_or_path=str(T5_DIR)),
    SummarizationDataArguments(),
    seq2seq_arguments(SUMMARIZATION, output_dir="/content/eval-t5", per_device_eval_batch_size=16),
)

t5_metrics = t5_trainer.evaluate(t5_trainer.test_dataset, metric_key_prefix="test")
for name, value in t5_metrics.items():
    if isinstance(value, float):
        print(f"{name:<25} {value:.4f}")

GPT-2 reports no task metric — the `Trainer` gives back `test_loss` and perplexity is `exp` of it,
which is why `gpt2.train.build_trainer` passes no `compute_metrics` at all. Its pad token is set to the
EOS token by `load_tokenizer` during training and written into `tokenizer_config.json`, so it
round-trips through Drive without any fixing up on reload.

In [ ]:
from headlines.gpt2.config import CLM, ClmDataArguments, ClmModelArguments
from headlines.gpt2.metrics import perplexity
from headlines.gpt2.train import build_trainer as build_gpt2_trainer


GPT2_DIR = Path("/content/drive/MyDrive/bert-t5-gpt2/gpt2")

gpt2_trainer = build_gpt2_trainer(
    ClmModelArguments(model_name_or_path=str(GPT2_DIR)),
    ClmDataArguments(),
    training_arguments(CLM, output_dir="/content/eval-gpt2", per_device_eval_batch_size=8),
)

gpt2_metrics = gpt2_trainer.evaluate(gpt2_trainer.test_dataset, metric_key_prefix="test")
print(f"test_loss:  {gpt2_metrics['test_loss']:.4f}")
print(f"perplexity: {perplexity(gpt2_metrics['test_loss']):.4f}")